# EDA — Alquiler de inmuebles en Lima

Análisis exploratorio interactivo del dataset de Properati Lima (Zenodo 10.5281/zenodo.7846211, CC-BY-4.0).

Este notebook reutiliza los módulos de `src/` para no duplicar lógica. Ejecútalo desde la raíz del repo con el entorno virtual activado.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from src import config as C, data_prep, eda

# Genera el dataset limpio si aún no existe
if not C.DATA_PROCESSED.exists():
    data_prep.construir_dataset()
df = pd.read_csv(C.DATA_PROCESSED)
print(df.shape)
df.head()

(867, 8)


,precio_pen,area_m2,dormitorios,banos,antiguedad,distrito,tipo_vivienda,moneda_original
0,3375.0,103.0,2.0,2.0,13.0,Miraflores,Apartamento,USD
1,2700.0,66.0,2.0,2.0,3.0,Barranco,Apartamento,USD
2,2600.0,60.0,1.0,2.0,NaN,Lince,Apartamento,PEN
3,2300.0,50.0,2.0,2.0,1.0,Santiago de Surco,Apartamento,PEN
4,8625.0,194.0,5.0,3.0,61.0,Chorrillos,Casa,USD


## Estadísticos descriptivos

In [2]:
df[C.NUM_FEATURES + [C.TARGET]].describe().round(1)

,area_m2,dormitorios,banos,antiguedad,precio_pen
count,867.0,867.0,867.0,684.0,867.0
mean,146.1,2.6,2.3,12.8,4172.2
std,150.2,1.3,1.3,13.2,3737.2
min,15.0,1.0,1.0,0.0,550.0
25%,69.5,2.0,2.0,4.0,2200.0
50%,105.0,3.0,2.0,9.0,3187.5
75%,165.0,3.0,3.0,18.0,4875.0
max,1500.0,12.0,15.0,120.0,37500.0


In [3]:
# Proporción de avisos por moneda original
df['moneda_original'].value_counts(normalize=True).round(3) * 100

moneda_original
USD    61.6
PEN    38.4
Name: proportion, dtype: float64

## Figuras
Regenera todas las figuras del EDA (se guardan en `reports/figures/`).

In [4]:
eda.graficos(df)


=== GENERANDO FIGURAS ===


  figura -> 01_distribucion_precio.png
  figura -> 02_precio_por_tipo.png


  figura -> 03_precio_por_distrito.png


  figura -> 04_area_vs_precio.png


  figura -> 05_correlacion.png

Correlación de cada feature con el precio:
area_m2        0.634
banos          0.503
dormitorios    0.463
antiguedad     0.149
Name: precio_pen, dtype: float64


In [5]:
# Correlación de cada feature con el precio
df[C.NUM_FEATURES + [C.TARGET]].corr()[C.TARGET].drop(C.TARGET).sort_values(ascending=False).round(3)

area_m2        0.634
banos          0.503
dormitorios    0.463
antiguedad     0.149
Name: precio_pen, dtype: float64

## Conclusiones del EDA

- El **precio está sesgado a la derecha** → se modela `log(1+precio)`.
- El **área** es la variable más correlacionada con el precio (r ≈ 0.63).
- Los distritos premium (Miraflores, San Isidro, Barranco, Surco) concentran los alquileres más altos.

Siguiente paso: `python -m src.train` y `python -m src.experiments`.